Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Landing zone, workflow 1: setup

Turns every folder of a **landing zone** in OCI Object Storage into a pair of tables:

| For each folder | What it is |
|---|---|
| an **external table** in `EXTERNAL_CATALOG` | points at the folder; nothing is copied, and dropping it leaves the files alone |
| a **managed Delta table** in `MANAGED_CATALOG` | where workflow 2 copies the rows; it records its source in the table property `landing_zone.source` |

**The folder convention.** Under `SOURCE_ROOT`, each top-level folder is one table. A folder may
contain `column=value/` subfolders, which become partition columns:

```
oci://<bucket>@<namespace>/landing/      <- SOURCE_ROOT
  sales/sale_date=2026-10-18/*.parquet   -> table sales, partitioned by sale_date
  customers/*.parquet                    -> table customers
  Web-Events/*.parquet                   -> table web_events (name sanitized)
  _staging/                              -> skipped: hidden folder
  docs/                                  -> skipped: no Parquet files
```

**Safe to rerun.** Tables that already exist are left alone, so run this again whenever a new folder
appears: only the new folder gets tables. It never creates catalogs, and never changes or drops a
table it did not create.

## 1 · Parameters

Set them here, or as job parameters on the workflow task (a job parameter wins).

In [ ]:
import re
from collections import Counter

def param(name, default):
    """Return the workflow job parameter `name` when it is set, otherwise `default`."""
    try:
        value = oidlUtils.parameters.getParameter(name, default)   # oidlUtils comes with the AIDP runtime
    except NameError:                                             # running interactively, not as a job
        value = default
    return default if value in (None, "") else value

def is_valid_name(name):
    """True for a plain catalog, schema or table name: letters, digits and _, not starting with a digit."""
    return re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name) is not None

def first_line(error):
    """The first useful line of an error message - Spark errors can run to pages."""
    lines = [line.strip(" :") for line in str(error).splitlines() if line.strip(" :")]
    # Errors raised in Java start with a generic "An error occurred while calling ..." line: skip it.
    useful = [line for line in lines if not line.startswith("An error occurred while calling")]
    return (useful or lines or [repr(error)])[0][:200]

SOURCE_ROOT      = param("SOURCE_ROOT", "oci://<bucket>@<namespace>/landing/")   # one folder per table
EXTERNAL_CATALOG = param("EXTERNAL_CATALOG", "default")   # catalog for the external tables
MANAGED_CATALOG  = param("MANAGED_CATALOG", "silver")     # catalog for the managed Delta tables
SCHEMA           = param("SCHEMA", "landing_zone_demo")   # the same schema name is used in both catalogs

# Check the parameters before touching anything.
SOURCE_ROOT = SOURCE_ROOT.rstrip("/") + "/"
if "<" in SOURCE_ROOT or not re.fullmatch(r"oci://[^@/'\\]+@[^/'\\]+/[^'\\]+", SOURCE_ROOT):
    raise ValueError("Set SOURCE_ROOT to a folder in your bucket, e.g. oci://my-bucket@my-namespace/landing/ "
                     "- it must include a prefix (the bucket root itself cannot be listed), "
                     "and no quotes or backslashes.")
for name in (EXTERNAL_CATALOG, MANAGED_CATALOG, SCHEMA):
    if not is_valid_name(name):
        raise ValueError(f"{name!r} is not a valid catalog or schema name")
if EXTERNAL_CATALOG == MANAGED_CATALOG:
    raise ValueError("EXTERNAL_CATALOG and MANAGED_CATALOG must be different: "
                     "the two tables of a folder get the same name.")

## 2 · Check the catalogs

The sample never creates catalogs - only the schema inside them. `SHOW SCHEMAS IN <catalog>` works on
any catalog that exists, and fails on one that doesn't, which makes it a simple existence check.

In [ ]:
def catalog_exists(catalog):
    try:
        spark.sql(f"SHOW SCHEMAS IN {catalog}").collect()
        return True
    except Exception as error:
        if "NOT_FOUND" in str(error):     # a catalog that does not exist raises SCHEMA_NOT_FOUND
            return False
        raise                             # anything else (no permission, a timeout) is a real error

missing_catalogs = [c for c in (EXTERNAL_CATALOG, MANAGED_CATALOG) if not catalog_exists(c)]
if missing_catalogs:
    raise RuntimeError(f"Catalog(s) not found: {missing_catalogs}. Create them first, or point "
                       "EXTERNAL_CATALOG / MANAGED_CATALOG at catalogs that exist.")

for catalog in (EXTERNAL_CATALOG, MANAGED_CATALOG):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{SCHEMA}")
print(f"schema {SCHEMA} is ready in {EXTERNAL_CATALOG} and in {MANAGED_CATALOG}")

## 3 · Find the table folders

Each top-level folder of `SOURCE_ROOT` is checked in this order:

1. **hidden** (the name starts with `_` or `.`) → skipped
2. **a quote or backslash in the name** → skipped: a table location can't contain them
3. **holds a Delta or Iceberg table** (a `_delta_log/` or `metadata/` subfolder) → skipped: out of scope here
4. **no `.parquet` file** anywhere inside → skipped
5. otherwise it is a table, named after the folder: lowercase, any character other than a letter, digit
   or `_` becomes `_`, and a leading digit gets a `t_` prefix. If two folders map to the same name,
   both are skipped - picking one would be a guess.

The listing uses Hadoop's FileSystem API, which reads Object Storage with the cluster's own identity.

In [ ]:
Path = spark._jvm.org.apache.hadoop.fs.Path
fs = Path(SOURCE_ROOT).getFileSystem(spark._jsc.hadoopConfiguration())

def is_hidden(name):
    return name.startswith(("_", "."))      # Spark ignores files and folders named like this

def first_parquet_file(folder):
    """The path of the first visible .parquet file under `folder`, relative to it - for example
    'sale_date=2026-10-18/part-00000.parquet' - or None if there is none."""
    files = fs.listFiles(Path(folder), True)            # True = look in subfolders too
    while files.hasNext():
        relative_path = files.next().getPath().toString()[len(folder) + 1:]
        parts = relative_path.split("/")                # ['sale_date=2026-10-18', 'part-00000.parquet']
        if relative_path.endswith(".parquet") and not any(is_hidden(part) for part in parts):
            return relative_path
    return None

def partition_columns(relative_path):
    """The column=value folders on the way to a file are its partitions:
    'sale_date=2026-10-18/part-00000.parquet'  ->  ['sale_date']"""
    return [part.split("=", 1)[0] for part in relative_path.split("/")[:-1] if "=" in part]

def table_name_for(folder_name):
    """Web-Events -> web_events,  2026_orders -> t_2026_orders"""
    name = re.sub(r"[^a-z0-9_]", "_", folder_name.lower())
    return "t_" + name if name[0].isdigit() else name

In [ ]:
found = []      # folders that hold a table: (folder name, folder path, table name, partition columns)
skipped = []    # (folder name, reason)

for entry in fs.listStatus(Path(SOURCE_ROOT)):
    if not entry.isDirectory():
        continue                                         # a loose file at the top level is not a table
    folder_name, folder = entry.getPath().getName(), entry.getPath().toString()
    if is_hidden(folder_name):
        skipped.append((folder_name, "hidden folder"))
        continue
    if "'" in folder_name or "\\" in folder_name:
        skipped.append((folder_name, "a quote or backslash in the name"))
        continue
    subfolders = {child.getPath().getName() for child in fs.listStatus(entry.getPath())}
    if "_delta_log" in subfolders or "metadata" in subfolders:
        skipped.append((folder_name, "Delta or Iceberg table, out of scope"))
        continue
    parquet_file = first_parquet_file(folder)
    if parquet_file is None:
        skipped.append((folder_name, "no .parquet files"))
        continue
    found.append((folder_name, folder, table_name_for(folder_name), partition_columns(parquet_file)))

# Two folders that map to the same table name are both skipped.
folders_per_name = Counter(table for _, _, table, _ in found)
tables = []     # the folders to set up: (folder name, folder path, table name, partition columns)
for folder_name, folder, table, partitions in found:
    if folders_per_name[table] > 1:
        skipped.append((folder_name, f"name collision: more than one folder maps to {table}"))
    else:
        tables.append((folder_name, folder, table, partitions))

print(f"{len(tables)} table folder(s), {len(skipped)} skipped")

## 4 · Create the tables

Two things here are not obvious:

- **The external table gets an explicit column list**, read from the Parquet files. On AIDP,
  `CREATE TABLE ... LOCATION` *without* columns succeeds but registers a table with **no columns**,
  so the notebook also checks the new table has some.
- **Tables are created only after checking they don't exist**, with explicit columns, rather than with
  `CREATE TABLE IF NOT EXISTS ... AS SELECT`, which can rebuild a table that already exists.

A table that already exists is never changed: if it points at another folder, or carries a different
`landing_zone.source`, it belongs to something else, and it is only reported.

In [ ]:
def columns_ddl(dataframe):
    """The columns of a DataFrame as SQL, e.g. "order_id BIGINT, amount DOUBLE, sale_date DATE"."""
    return dataframe._jdf.schema().toDDL()

def table_location(full_name):
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {full_name}").collect()
    return next((row.data_type for row in rows if row.col_name == "Location"), "").rstrip("/")

def landing_zone_source(full_name):
    """The value of the landing_zone.source property, or None if the table does not have it."""
    properties = {row.key: row.value for row in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    return properties.get("landing_zone.source")

def setup_external_table(table, folder, partitions):
    """Create the external table over `folder` unless it exists. Returns what was done."""
    external_table = f"{EXTERNAL_CATALOG}.{SCHEMA}.{table}"
    if spark.catalog.tableExists(external_table):
        location = table_location(external_table)
        return "exists" if location == folder else f"skipped: exists, but points to {location}"
    columns = columns_ddl(spark.read.parquet(folder))       # Parquet files carry their schema
    partitioned_by = f"PARTITIONED BY ({', '.join(f'`{c}`' for c in partitions)})" if partitions else ""
    spark.sql(f"CREATE TABLE {external_table} ({columns}) USING PARQUET {partitioned_by} LOCATION '{folder}'")
    if not spark.table(external_table).columns:
        spark.sql(f"DROP TABLE {external_table}")   # external: the files stay, and the next run tries again
        raise RuntimeError(f"{external_table} was created without columns")
    return "created"

def setup_managed_table(table):
    """Create the managed Delta table for this folder unless it exists. Returns what was done."""
    external_table = f"{EXTERNAL_CATALOG}.{SCHEMA}.{table}"
    managed_table = f"{MANAGED_CATALOG}.{SCHEMA}.{table}"
    if spark.catalog.tableExists(managed_table):
        current_source = landing_zone_source(managed_table)
        if current_source == external_table:
            return "exists"
        if current_source is None:
            return "skipped: exists without the landing_zone.source property"
        return f"skipped: exists, but its landing_zone.source is {current_source}"
    columns = columns_ddl(spark.table(external_table))
    spark.sql(f"CREATE TABLE {managed_table} ({columns}, _source_file STRING, _ingested_at TIMESTAMP) "
              f"USING DELTA TBLPROPERTIES ('landing_zone.source' = '{external_table}')")
    return "created"

In [ ]:
summary = []        # (folder, external table, external action, managed table, managed action)

for folder_name, folder, table, partitions in tables:
    try:
        external_action = setup_external_table(table, folder, partitions)
    except Exception as error:      # one bad folder must not stop the others
        external_action = "failed: " + first_line(error)

    if external_action in ("created", "exists"):
        try:
            managed_action = setup_managed_table(table)
        except Exception as error:
            managed_action = "failed: " + first_line(error)
    else:
        managed_action = "skipped: the external table is not usable"

    summary.append((folder_name, f"{EXTERNAL_CATALOG}.{SCHEMA}.{table}", external_action,
                    f"{MANAGED_CATALOG}.{SCHEMA}.{table}", managed_action))

for folder_name, reason in skipped:
    summary.append((folder_name, "-", "skipped: " + reason, "-", "-"))

spark.createDataFrame(summary, "folder STRING, external_table STRING, external STRING, "
                               "managed_table STRING, managed STRING").show(100, truncate=False)

failures = [row[0] for row in summary if row[2].startswith("failed") or row[4].startswith("failed")]
if failures:
    raise RuntimeError(f"{len(failures)} folder(s) failed: {failures} - see the summary above")